# GPU Matrix-Matrix Multiplication

This notebook demonstrates matrix multiplication using CUDA C++ in Google Colab.
We'll implement three versions:
1. CPU Implementation
2. Naive CUDA Implementation
3. cuBLAS Implementation
4. Tensor and Half Precision Implementation
5. Tiled/Shared memory optimization


## Setup: Install CUDA Toolkit (if needed)

In [16]:
# Check CUDA availability
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## 1. CPU Matrix Multiplication

In [46]:
%%writefile matmul_cpu.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void matmul_cpu(float *A, float *B, float *C,
                int M, int N, int K)
{
    for(int i=0;i<M;i++)
    {
        for(int j=0;j<K;j++)
        {
            float sum=0.0f;

            for(int k=0;k<N;k++)
            {
                sum += A[i*N+k]*B[k*K+j];
            }

            C[i*K+j]=sum;
        }
    }
}

void init_matrix(float *mat,int size)
{
    for(int i=0;i<size;i++)
        mat[i]=(float)(rand()%5);
}

int main()
{
    int M=1024,N=1024,K=1024;

    size_t bytes_A=M*N*sizeof(float);
    size_t bytes_B=N*K*sizeof(float);
    size_t bytes_C=M*K*sizeof(float);

    float *A=(float*)malloc(bytes_A);
    float *B=(float*)malloc(bytes_B);
    float *C=(float*)malloc(bytes_C);

    init_matrix(A,M*N);
    init_matrix(B,N*K);

    clock_t start,end;

    start=clock();

    matmul_cpu(A,B,C,M,N,K);

    end=clock();

    double time_ms=((double)(end-start)/CLOCKS_PER_SEC)*1000;

    double flops=2.0*M*N*K;
    double gflops=(flops/(time_ms/1000.0))/1e9;

    printf("CPU Time: %f ms\n",time_ms);
    printf("Performance: %f GFLOPS\n",gflops);

    free(A);free(B);free(C);

    return 0;
}

Overwriting matmul_cpu.c


In [47]:
!gcc matmul_cpu.c -O3 -o cpu
!./cpu

CPU Time: 3236.712000 ms
Performance: 0.663477 GFLOPS


## 2. Naive CUDA Matrix Multiplication

In [38]:
%%writefile matmul_naive.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void matmul_naive(float *A, float *B, float *C,
                             int M, int N, int K)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if(row < M && col < K)
    {
        float sum = 0.0f;

        for(int i = 0; i < N; i++)
        {
            sum += A[row * N + i] * B[i * K + col];
        }

        C[row * K + col] = sum;
    }
}

void init_matrix(float *mat, int size)
{
    for(int i=0;i<size;i++)
        mat[i] = (float)(rand()%5);
}

int main()
{
    int M = 1024, N = 1024, K = 1024;

    size_t bytes_A = M*N*sizeof(float);
    size_t bytes_B = N*K*sizeof(float);
    size_t bytes_C = M*K*sizeof(float);

    float *h_A=(float*)malloc(bytes_A);
    float *h_B=(float*)malloc(bytes_B);
    float *h_C=(float*)malloc(bytes_C);

    init_matrix(h_A,M*N);
    init_matrix(h_B,N*K);

    float *d_A,*d_B,*d_C;
    cudaMalloc(&d_A,bytes_A);
    cudaMalloc(&d_B,bytes_B);
    cudaMalloc(&d_C,bytes_C);

    cudaMemcpy(d_A,h_A,bytes_A,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,bytes_B,cudaMemcpyHostToDevice);

    dim3 threads(16,16);
    dim3 blocks((K+15)/16,(M+15)/16);

    cudaEvent_t start,stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matmul_naive<<<blocks,threads>>>(d_A,d_B,d_C,M,N,K);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms=0;
    cudaEventElapsedTime(&ms,start,stop);

    double flops = 2.0*M*N*K;
    double gflops = (flops/(ms/1000.0))/1e9;

    printf("Naive (No Tile) Time: %f ms\n",ms);
    printf("Performance: %f GFLOPS\n",gflops);

    cudaMemcpy(h_C,d_C,bytes_C,cudaMemcpyDeviceToHost);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);

    return 0;
}

Overwriting matmul_naive.cu


In [39]:
# Compile and run
!nvcc matmul_naive.cu -o matmul_naive
!./matmul_naive

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Naive (No Tile) Time: 8.754432 ms
Performance: 245.302461 GFLOPS


## 3. Comparison with cuBLAS

In [41]:
%%writefile matmul_cublas.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cublas_v2.h>

void init_matrix(float *mat, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        mat[i] = (float)(rand() % 100) / 10.0f;
    }
}

int main() {
    int M = 1024, N = 1024, K = 1024;
    size_t bytes_A = M * N * sizeof(float);
    size_t bytes_B = N * K * sizeof(float);
    size_t bytes_C = M * K * sizeof(float);

    float *h_A = (float*)malloc(bytes_A);
    float *h_B = (float*)malloc(bytes_B);
    float *h_C = (float*)malloc(bytes_C);

    init_matrix(h_A, M, N);
    init_matrix(h_B, N, K);

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes_A);
    cudaMalloc(&d_B, bytes_B);
    cudaMalloc(&d_C, bytes_C);

    cudaMemcpy(d_A, h_A, bytes_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes_B, cudaMemcpyHostToDevice);

    cublasHandle_t handle;
    cublasCreate(&handle);

    float alpha = 1.0f, beta = 0.0f;

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    // C = alpha * A * B + beta * C
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, K, M, N, &alpha, d_B, K, d_A, N, &beta, d_C, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("cuBLAS MatMul: %.3f ms\n", milliseconds);
    printf("Performance: %.2f GFLOPS\n", (2.0 * M * N * K) / (milliseconds * 1e6));

    cudaMemcpy(h_C, d_C, bytes_C, cudaMemcpyDeviceToHost);

    cublasDestroy(handle);
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);

    return 0;
}

Overwriting matmul_cublas.cu


In [42]:
# Compile and run with cuBLAS
!nvcc matmul_cublas.cu -o matmul_cublas -lcublas
!./matmul_cublas

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
cuBLAS MatMul: 6.038 ms
Performance: 355.68 GFLOPS


## 4. Optimized Tensor Matrix Multiplication with Shared Memory

In [35]:
%%writefile tensor_core_gemm.cu
#include <stdio.h>
#include <cuda.h>
#include <mma.h>
#include <cuda_fp16.h>

using namespace nvcuda;

#define M 1024
#define N 1024
#define K 1024

#define WMMA_M 16
#define WMMA_N 16
#define WMMA_K 16

__global__ void tensor_gemm(half *A, half *B, float *C)
{
    int warpM = (blockIdx.x * blockDim.x + threadIdx.x) / 32;
    int warpN = (blockIdx.y * blockDim.y + threadIdx.y);

    if (warpM * WMMA_M >= M || warpN * WMMA_N >= K) return;

    wmma::fragment<wmma::matrix_a, WMMA_M, WMMA_N, WMMA_K, half, wmma::row_major> a_frag;
    wmma::fragment<wmma::matrix_b, WMMA_M, WMMA_N, WMMA_K, half, wmma::col_major> b_frag;
    wmma::fragment<wmma::accumulator, WMMA_M, WMMA_N, WMMA_K, float> c_frag;

    wmma::fill_fragment(c_frag, 0.0f);

    for(int i=0;i<N;i+=WMMA_K)
    {
        int aRow = warpM * WMMA_M;
        int aCol = i;

        int bRow = i;
        int bCol = warpN * WMMA_N;

        wmma::load_matrix_sync(a_frag, A + aRow * N + aCol, N);
        wmma::load_matrix_sync(b_frag, B + bRow * K + bCol, K);

        wmma::mma_sync(c_frag, a_frag, b_frag, c_frag);
    }

    int cRow = warpM * WMMA_M;
    int cCol = warpN * WMMA_N;

    wmma::store_matrix_sync(C + cRow * K + cCol, c_frag, K, wmma::mem_row_major);
}

void init_matrix(half *mat, int size)
{
    for(int i=0;i<size;i++)
        mat[i] = __float2half((float)(rand()%5));
}

int main()
{
    size_t bytes_A = M*N*sizeof(half);
    size_t bytes_B = N*K*sizeof(half);
    size_t bytes_C = M*K*sizeof(float);

    half *h_A=(half*)malloc(bytes_A);
    half *h_B=(half*)malloc(bytes_B);
    float *h_C=(float*)malloc(bytes_C);

    init_matrix(h_A,M*N);
    init_matrix(h_B,N*K);

    half *d_A,*d_B;
    float *d_C;

    cudaMalloc(&d_A,bytes_A);
    cudaMalloc(&d_B,bytes_B);
    cudaMalloc(&d_C,bytes_C);

    cudaMemcpy(d_A,h_A,bytes_A,cudaMemcpyHostToDevice);
    cudaMemcpy(d_B,h_B,bytes_B,cudaMemcpyHostToDevice);

    dim3 threads(128,4);
    dim3 blocks((M/WMMA_M)/4,(K/WMMA_N));

    cudaEvent_t start,stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    tensor_gemm<<<blocks,threads>>>(d_A,d_B,d_C);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float ms=0;
    cudaEventElapsedTime(&ms,start,stop);

    double flops=2.0*M*N*K;
    double gflops=(flops/(ms/1000.0))/1e9;

    printf("Tensor Core MatMul: %f ms\n",ms);
    printf("Performance: %f GFLOPS\n",gflops);

    cudaMemcpy(h_C,d_C,bytes_C,cudaMemcpyDeviceToHost);

    return 0;
}

Overwriting tensor_core_gemm.cu


In [36]:
!nvcc -arch=sm_75 tensor_core_gemm.cu -o tensor
!./tensor

Tensor Core MatMul: 1.469792 ms
Performance: 1461.079959 GFLOPS


## 5. Optimized Tiled Matrix Multiplication with Shared Memory

In [22]:
%%writefile matmul_tiled.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define TILE_SIZE 16

// Tiled kernel using shared memory
__global__ void matmul_tiled(float *A, float *B, float *C, int M, int N, int K) {
    __shared__ float tile_A[TILE_SIZE][TILE_SIZE];
    __shared__ float tile_B[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * TILE_SIZE + threadIdx.y;
    int col = blockIdx.x * TILE_SIZE + threadIdx.x;

    float sum = 0.0f;

    // Loop over tiles
    for (int t = 0; t < (N + TILE_SIZE - 1) / TILE_SIZE; t++) {
        // Load tiles into shared memory
        if (row < M && t * TILE_SIZE + threadIdx.x < N)
            tile_A[threadIdx.y][threadIdx.x] = A[row * N + t * TILE_SIZE + threadIdx.x];
        else
            tile_A[threadIdx.y][threadIdx.x] = 0.0f;

        if (t * TILE_SIZE + threadIdx.y < N && col < K)
            tile_B[threadIdx.y][threadIdx.x] = B[(t * TILE_SIZE + threadIdx.y) * K + col];
        else
            tile_B[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();

        // Compute partial sum
        for (int i = 0; i < TILE_SIZE; i++) {
            sum += tile_A[threadIdx.y][i] * tile_B[i][threadIdx.x];
        }

        __syncthreads();
    }

    if (row < M && col < K) {
        C[row * K + col] = sum;
    }
}

void init_matrix(float *mat, int rows, int cols) {
    for (int i = 0; i < rows * cols; i++) {
        mat[i] = (float)(rand() % 100) / 10.0f;
    }
}

int main() {
    int M = 1024, N = 1024, K = 1024;
    size_t bytes_A = M * N * sizeof(float);
    size_t bytes_B = N * K * sizeof(float);
    size_t bytes_C = M * K * sizeof(float);

    float *h_A = (float*)malloc(bytes_A);
    float *h_B = (float*)malloc(bytes_B);
    float *h_C = (float*)malloc(bytes_C);

    init_matrix(h_A, M, N);
    init_matrix(h_B, N, K);

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes_A);
    cudaMalloc(&d_B, bytes_B);
    cudaMalloc(&d_C, bytes_C);

    cudaMemcpy(d_A, h_A, bytes_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes_B, cudaMemcpyHostToDevice);

    dim3 threads(TILE_SIZE, TILE_SIZE);
    dim3 blocks((K + TILE_SIZE - 1) / TILE_SIZE, (M + TILE_SIZE - 1) / TILE_SIZE);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);
    matmul_tiled<<<blocks, threads>>>(d_A, d_B, d_C, M, N, K);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float milliseconds = 0;
    cudaEventElapsedTime(&milliseconds, start, stop);

    printf("Tiled CUDA MatMul: %.3f ms\n", milliseconds);
    printf("Performance: %.2f GFLOPS\n", (2.0 * M * N * K) / (milliseconds * 1e6));

    cudaMemcpy(h_C, d_C, bytes_C, cudaMemcpyDeviceToHost);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);

    return 0;
}

Writing matmul_tiled.cu


In [23]:
# Compile and run
!nvcc matmul_tiled.cu -o matmul_tiled
!./matmul_tiled

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Tiled CUDA MatMul: 5.291 ms
Performance: 405.86 GFLOPS


## 6. Performance Comparison

In [48]:
import matplotlib.pyplot as plt
import numpy as np

# Run all three versions and collect timings
print("Running all versions...\n")
print("1. CPU implementation:")
!./cpu
print("\n3. Naive implementation:")
!./matmul_naive
print("\n4. cuBLAS implementation:")
!./matmul_cublas
print("\n4. Tensor implementation:")
!./tensor
print("\n5. Tiled implementation:")
!./matmul_tiled

Running all versions...

1. CPU implementation:
CPU Time: 3138.404000 ms
Performance: 0.684260 GFLOPS

3. Naive implementation:
Naive (No Tile) Time: 7.939808 ms
Performance: 270.470480 GFLOPS

4. cuBLAS implementation:
cuBLAS MatMul: 7.547 ms
Performance: 284.56 GFLOPS

4. Tensor implementation:
Tensor Core MatMul: 0.925280 ms
Performance: 2320.901464 GFLOPS

5. Tiled implementation:
Tiled CUDA MatMul: 2.787 ms
Performance: 770.67 GFLOPS
